# 📓 Notebook 03 — Test Full OCR Pipeline

This notebook tests the complete pipeline on a real handwritten Sinhala sentence photo.

**Run order:**
1. Run Notebook 01 (Explore Dataset) first
2. Run Notebook 02 (Train Model) and confirm model is saved
3. Run this notebook

**What this notebook does:**
- Loads your trained model
- Loads a test photo of handwritten Sinhala
- Runs full pipeline: greyscale → denoise → binarize → deskew → segment → predict → sentence
- Shows every intermediate step visually
- Displays final recognized text

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — Configuration
# ════════════════════════════════════════════════════════════

import os

# ── Path to trained model (from Notebook 02 output) ──────────
# When you add Notebook 02 output as dataset input, it appears here:
MODEL_PATH    = '/kaggle/input/02-train-cnn-model/models/sinhala_cnn.h5'
LABELMAP_PATH = '/kaggle/input/02-train-cnn-model/models/label_map.json'

# ── Path to your test image ───────────────────────────────────
# Upload a photo of handwritten Sinhala using the Kaggle file upload
# (left sidebar → folder icon → Upload)
# Then set the correct filename below:
TEST_IMAGE_PATH = '/kaggle/input/test-images/test_sentence.jpg'

# ── Model settings (must match training settings) ─────────────
IMAGE_SIZE = (64, 64)

# ── Output folder ─────────────────────────────────────────────
OUTPUT_DIR = '/kaggle/working/test_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✅ Configuration set')
print(f'   Model path  : {MODEL_PATH}')
print(f'   Label map   : {LABELMAP_PATH}')
print(f'   Test image  : {TEST_IMAGE_PATH}')
print(f'   Output dir  : {OUTPUT_DIR}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — Imports
# ════════════════════════════════════════════════════════════

import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict
import tensorflow as tf
from tensorflow.keras.models import load_model

print(f'✅ TensorFlow : {tf.__version__}')
print(f'   GPU        : {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — Load trained model and label map
# ════════════════════════════════════════════════════════════

# Load CNN model
model = load_model(MODEL_PATH)
print(f'✅ Model loaded')
print(f'   Input shape  : {model.input_shape}')
print(f'   Output shape : {model.output_shape}')
print(f'   Parameters   : {model.count_params():,}')

# Load label map
with open(LABELMAP_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)
label_map = {int(k): v for k, v in raw.items()}

NUM_CLASSES = len(label_map)
print(f'\n✅ Label map loaded : {NUM_CLASSES} Sinhala characters')
print(f'   Sample mappings:')
for k, v in list(label_map.items())[:8]:
    print(f'   [{k}] → "{v}"')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — Load and display test image
# ════════════════════════════════════════════════════════════

original = cv2.imread(TEST_IMAGE_PATH)
if original is None:
    raise FileNotFoundError(
        f'Cannot read image at: {TEST_IMAGE_PATH}\n'
        'Upload your test image using the Kaggle file upload panel (left sidebar)'
    )

original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)
h, w = original.shape[:2]

plt.figure(figsize=(14, 6))
plt.imshow(original_rgb)
plt.title(f'Original Photo — {w}×{h} pixels', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/00_original.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'✅ Image loaded: {w}×{h} pixels')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — Preprocessing pipeline (all steps shown)
# ════════════════════════════════════════════════════════════

# ── Step A: Convert to greyscale ─────────────────────────────
grey = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)

# ── Step B: Denoise ──────────────────────────────────────────
denoised = cv2.GaussianBlur(grey, (3, 3), 0)

# ── Step C: Adaptive binarization ────────────────────────────
# Adaptive threshold handles shadows and uneven lighting from phone camera
adaptive = cv2.adaptiveThreshold(
    denoised, 255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV,
    blockSize=31,
    C=10
)

# Also compute Otsu for comparison
_, otsu = cv2.threshold(
    denoised, 0, 255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)

# Choose best binarization
adaptive_density = np.sum(adaptive > 0) / adaptive.size
binary_raw = adaptive if adaptive_density < 0.40 else otsu
print(f'Binarization method: {"Adaptive" if adaptive_density < 0.40 else "Otsu"}')
print(f'Foreground pixel density: {adaptive_density*100:.1f}%')

# ── Step D: Remove noise blobs ────────────────────────────────
kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
cleaned = cv2.morphologyEx(binary_raw, cv2.MORPH_OPEN,  kernel, iterations=1)

# ── Step E: Close broken strokes ─────────────────────────────
closed  = cv2.morphologyEx(cleaned,    cv2.MORPH_CLOSE, kernel, iterations=1)

# ── Step F: Deskew ────────────────────────────────────────────
coords = np.column_stack(np.where(closed > 0))
angle  = 0.0
if len(coords) >= 50:
    rect  = cv2.minAreaRect(coords)
    angle = rect[-1]
    if angle < -45:
        angle = 90 + angle
    if abs(angle) >= 0.5:
        hh, ww  = closed.shape
        centre  = (ww // 2, hh // 2)
        rot_mat = cv2.getRotationMatrix2D(centre, angle, 1.0)
        binary  = cv2.warpAffine(closed, rot_mat, (ww, hh),
                                  flags=cv2.INTER_LINEAR,
                                  borderMode=cv2.BORDER_CONSTANT,
                                  borderValue=0)
        print(f'Deskewed by {angle:.2f}°')
    else:
        binary = closed
        print(f'No deskewing needed (angle={angle:.2f}°)')
else:
    binary = closed
    print('Not enough content to estimate skew')

# ── Visualise all preprocessing steps ────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

steps = [
    (original_rgb,  'Step 1: Original Photo',           False),
    (grey,          'Step 2: Greyscale',                 True),
    (denoised,      'Step 3: Denoised',                  True),
    (adaptive,      'Step 4a: Adaptive Threshold',       True),
    (otsu,          'Step 4b: Otsu Threshold',           True),
    (cleaned,       'Step 5: Noise Removed',             True),
    (closed,        'Step 6: Strokes Closed',            True),
    (binary,        f'Step 7: Deskewed ({angle:.1f}°)',  True),
]

for ax, (img, title, is_grey) in zip(axes, steps):
    if is_grey:
        ax.imshow(img, cmap='gray')
    else:
        ax.imshow(img)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Preprocessing Pipeline — Every Step', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/01_preprocessing.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n✅ Preprocessing complete — binary image ready for segmentation')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — Line segmentation with projection profile
# ════════════════════════════════════════════════════════════

MIN_CHAR_HEIGHT = 10
V_PADDING       = 6

# Horizontal projection
h_proj    = np.sum(binary, axis=1).astype(np.float32)
threshold = max(1.0, float(np.max(h_proj)) * 0.04)

lines    = []
in_line  = False
start    = 0
H, W     = binary.shape

for i, val in enumerate(h_proj):
    if val > threshold and not in_line:
        start, in_line = i, True
    elif val <= threshold and in_line:
        y1 = max(0, start - V_PADDING)
        y2 = min(H, i   + V_PADDING)
        if (y2 - y1) >= MIN_CHAR_HEIGHT:
            lines.append((y1, y2))
        in_line = False

if in_line:
    lines.append((max(0, start - V_PADDING), H))

print(f'✅ Lines detected: {len(lines)}')
for i, (y1, y2) in enumerate(lines):
    print(f'   Line {i+1}: rows {y1} → {y2}  (height: {y2-y1}px)')

# Plot projection + line boundaries
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

ax1.plot(h_proj, np.arange(len(h_proj)), color='#3b82f6', linewidth=1.5)
ax1.axvline(x=threshold, color='red', linestyle='--', alpha=0.7, label=f'Threshold')
for y1, y2 in lines:
    ax1.axhspan(y1, y2, alpha=0.15, color='green')
ax1.invert_yaxis()
ax1.set_title('Horizontal Projection Profile', fontweight='bold')
ax1.set_xlabel('Pixel sum per row')
ax1.set_ylabel('Row index')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Show line boundaries on binary image
binary_vis = cv2.cvtColor(binary, cv2.COLOR_GRAY2RGB)
for i, (y1, y2) in enumerate(lines):
    cv2.rectangle(binary_vis, (0, y1), (W-1, y2), (0, 200, 100), 2)
    cv2.putText(binary_vis, f'L{i+1}', (5, y1 + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 100, 0), 2)

ax2.imshow(binary_vis)
ax2.set_title('Detected Text Lines (green boxes)', fontweight='bold')
ax2.axis('off')

plt.suptitle('Line Segmentation', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_line_segmentation.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — Character segmentation using connected components
# ════════════════════════════════════════════════════════════

MIN_CHAR_WIDTH = 8
MIN_CHAR_AREA  = 80
CHAR_PADDING   = 6
MERGE_GAP      = 5

def extract_characters(binary, lines):
    all_chars = []

    for line_idx, (y1, y2) in enumerate(lines):
        line_strip = binary[y1:y2, :]
        H_l, W_l   = line_strip.shape

        # Connected component analysis
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
            line_strip, connectivity=8
        )

        raw_chars = []
        for lbl in range(1, num_labels):
            x    = stats[lbl, cv2.CC_STAT_LEFT]
            y    = stats[lbl, cv2.CC_STAT_TOP]
            w_cc = stats[lbl, cv2.CC_STAT_WIDTH]
            h_cc = stats[lbl, cv2.CC_STAT_HEIGHT]
            area = stats[lbl, cv2.CC_STAT_AREA]

            if w_cc < MIN_CHAR_WIDTH or h_cc < MIN_CHAR_HEIGHT or area < MIN_CHAR_AREA:
                continue

            cx1 = max(0, x           - CHAR_PADDING)
            cx2 = min(W_l, x + w_cc  + CHAR_PADDING)
            cy1 = max(0,  y1 + y     - CHAR_PADDING)
            cy2 = min(binary.shape[0], y1 + y + h_cc + CHAR_PADDING)

            raw_chars.append({
                'image':    binary[cy1:cy2, cx1:cx2],
                'x1': cx1, 'x2': cx2,
                'y1': cy1, 'y2': cy2,
                'area':  area,
                'width': w_cc,
                'line_idx': line_idx,
            })

        # Sort left to right
        raw_chars.sort(key=lambda c: c['x1'])

        # Merge very close components (diacritics joining base chars)
        merged = []
        if raw_chars:
            merged = [raw_chars[0]]
            for ch in raw_chars[1:]:
                gap = ch['x1'] - merged[-1]['x2']
                if gap <= MERGE_GAP:
                    prev = merged[-1]
                    new_x1 = min(prev['x1'], ch['x1'])
                    new_x2 = max(prev['x2'], ch['x2'])
                    new_y1 = min(prev['y1'], ch['y1'])
                    new_y2 = max(prev['y2'], ch['y2'])
                    merged[-1] = {
                        'image':    binary[new_y1:new_y2, new_x1:new_x2],
                        'x1': new_x1, 'x2': new_x2,
                        'y1': new_y1, 'y2': new_y2,
                        'area':  prev['area'] + ch['area'],
                        'width': new_x2 - new_x1,
                        'line_idx': line_idx,
                    }
                else:
                    merged.append(ch)

        all_chars.extend(merged)
        print(f'   Line {line_idx+1}: {len(merged)} characters')

    return all_chars

print('Segmenting characters...')
all_chars = extract_characters(binary, lines)
print(f'\n✅ Total characters segmented: {len(all_chars)}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — Visualise all segmented characters
# ════════════════════════════════════════════════════════════

n    = len(all_chars)
cols = 12
rows = max(1, (n + cols - 1) // cols)

fig, axes = plt.subplots(rows, cols, figsize=(24, rows * 2.5))
axes = np.array(axes).flatten()

for i, ch in enumerate(all_chars):
    if i >= len(axes):
        break
    axes[i].imshow(ch['image'], cmap='gray')
    axes[i].set_title(
        f'#{i+1}\nL{ch["line_idx"]+1}',
        fontsize=8, pad=2
    )
    axes[i].axis('off')

for i in range(n, len(axes)):
    axes[i].axis('off')

plt.suptitle(
    f'All Segmented Characters — {n} total  |  {len(lines)} line(s)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_segmented_chars.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 9 — Predict every character using CNN
# ════════════════════════════════════════════════════════════

TOP_K = 3

def prepare_for_cnn(char_img):
    """Pad to square → resize → normalize → add dimensions"""
    h, w = char_img.shape[:2]
    size    = max(h, w)
    pad_top = (size - h) // 2
    pad_bot = size - h - pad_top
    pad_lft = (size - w) // 2
    pad_rgt = size - w - pad_lft
    padded  = cv2.copyMakeBorder(
        char_img, pad_top, pad_bot, pad_lft, pad_rgt,
        cv2.BORDER_CONSTANT, value=0
    )
    resized    = cv2.resize(padded, IMAGE_SIZE, interpolation=cv2.INTER_AREA)
    normalized = resized.astype('float32') / 255.0
    return normalized.reshape(1, IMAGE_SIZE[0], IMAGE_SIZE[1], 1)


print(f'🔍 Predicting {len(all_chars)} characters...')
print('─' * 60)

char_results = []
for i, ch in enumerate(all_chars):
    inp   = prepare_for_cnn(ch['image'])
    probs = model.predict(inp, verbose=0)[0]

    top_indices = np.argsort(probs)[::-1][:TOP_K]
    best_idx    = int(top_indices[0])
    best_letter = label_map.get(best_idx, '?')
    best_conf   = round(float(probs[best_idx]) * 100, 2)

    top_k = [
        {'letter': label_map.get(int(j), '?'),
         'confidence': round(float(probs[j]) * 100, 2)}
        for j in top_indices
    ]

    result = {
        **ch,
        'letter':     best_letter,
        'confidence': best_conf,
        'top_k':      top_k,
        'low_conf':   best_conf < 60.0,
    }
    char_results.append(result)

    # Confidence bar
    bar   = '█' * int(best_conf / 5)
    flag  = '⚠️ ' if best_conf < 60 else '  '
    print(f'  {flag}#{i+1:>3}  L{ch["line_idx"]+1}  "{best_letter}"  '
          f'{best_conf:5.1f}%  {bar}')

avg_conf = np.mean([c['confidence'] for c in char_results])
low_conf_count = sum(1 for c in char_results if c['low_conf'])

print('─' * 60)
print(f'✅ Predictions complete!')
print(f'   Average confidence      : {avg_conf:.1f}%')
print(f'   Low confidence (<60%)   : {low_conf_count} characters')
print(f'   High confidence (≥80%)  : {sum(1 for c in char_results if c["confidence"] >= 80)}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 10 — Assemble final sentence
# ════════════════════════════════════════════════════════════

WORD_GAP_RATIO = 1.8

lines_dict = defaultdict(list)
for ch in char_results:
    lines_dict[ch['line_idx']].append(ch)

sentence_lines = []
line_details   = []

for line_idx in sorted(lines_dict.keys()):
    line_chars = sorted(lines_dict[line_idx], key=lambda c: c['x1'])
    avg_w      = np.mean([c['width'] for c in line_chars]) if line_chars else 20

    line_text = ''
    for i, ch in enumerate(line_chars):
        if i == 0:
            line_text += ch['letter']
        else:
            gap = ch['x1'] - line_chars[i-1]['x2']
            if gap > avg_w * WORD_GAP_RATIO:
                line_text += ' '
            line_text += ch['letter']

    sentence_lines.append(line_text)
    line_details.append(line_chars)

full_text = '\n'.join(sentence_lines)

print('═' * 60)
print('  RECOGNIZED SINHALA TEXT')
print('═' * 60)
for i, line in enumerate(sentence_lines):
    print(f'  Line {i+1}: {line}')
print('─' * 60)
print(f'  Characters : {len(char_results)}')
print(f'  Lines      : {len(sentence_lines)}')
print(f'  Avg conf   : {avg_conf:.1f}%')
print('═' * 60)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 11 — Annotated result image with bounding boxes
# ════════════════════════════════════════════════════════════

annotated = original.copy()

for ch in char_results:
    # Color by confidence: green ≥80, orange 60-79, red <60
    if ch['confidence'] >= 80:
        color = (34, 197, 94)    # green
    elif ch['confidence'] >= 60:
        color = (245, 158, 11)   # orange
    else:
        color = (239, 68, 68)    # red

    cv2.rectangle(annotated,
                  (ch['x1'], ch['y1']),
                  (ch['x2'], ch['y2']),
                  color, 2)

annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

ax1.imshow(annotated_rgb)
ax1.set_title('Detected Characters\n🟢 ≥80%  🟡 60-79%  🔴 <60%',
              fontsize=12, fontweight='bold')
ax1.axis('off')

# Text result panel
ax2.set_facecolor('#f8fafc')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis('off')

ax2.text(0.5, 0.95, 'Recognized Text',
         ha='center', va='top', fontsize=14,
         fontweight='bold', color='#1e3a5f')
ax2.axhline(y=0.88, color='#e5e7eb', linewidth=1)

y_pos = 0.80
for i, line in enumerate(sentence_lines):
    ax2.text(0.05, y_pos, f'Line {i+1}:', ha='left', va='top',
             fontsize=10, color='#6b7280')
    ax2.text(0.05, y_pos - 0.06, line, ha='left', va='top',
             fontsize=16, fontweight='bold', color='#0f172a')
    y_pos -= 0.18

ax2.text(0.5, 0.12, f'Characters: {len(char_results)}   |   '
                    f'Avg Confidence: {avg_conf:.1f}%',
         ha='center', va='top', fontsize=11, color='#6b7280')

plt.suptitle('OCR Result', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_final_result.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 12 — Character confidence breakdown chart
# ════════════════════════════════════════════════════════════

letters = [c['letter']     for c in char_results]
confs   = [c['confidence'] for c in char_results]
colors  = ['#22c55e' if c >= 80 else '#f59e0b' if c >= 60 else '#ef4444'
           for c in confs]

fig, ax = plt.subplots(figsize=(max(12, len(letters) * 0.6), 5))

bars = ax.bar(range(len(letters)), confs, color=colors, width=0.7,
              edgecolor='white', linewidth=0.5)

# Letter labels on x-axis
ax.set_xticks(range(len(letters)))
ax.set_xticklabels(letters, fontsize=14, fontweight='bold')

# Confidence value on top of each bar
for bar, conf in zip(bars, confs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{conf:.0f}%',
        ha='center', va='bottom', fontsize=8, color='#374151'
    )

ax.axhline(y=80, color='#22c55e', linestyle='--', alpha=0.6, label='High (80%)')
ax.axhline(y=60, color='#f59e0b', linestyle='--', alpha=0.6, label='Medium (60%)')
ax.set_ylim(0, 110)
ax.set_ylabel('Confidence %', fontsize=12)
ax.set_title(
    f'Per-character Confidence  |  Avg: {avg_conf:.1f}%  '
    f'|  🟢 ≥80%: {sum(1 for c in confs if c>=80)}  '
    f'🟡 60-79%: {sum(1 for c in confs if 60<=c<80)}  '
    f'🔴 <60%: {sum(1 for c in confs if c<60)}',
    fontsize=12, fontweight='bold'
)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_confidence_chart.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 13 — Summary and output files
# ════════════════════════════════════════════════════════════

import json

# Save full result as JSON
output_data = {
    'text':           full_text,
    'lines':          sentence_lines,
    'char_count':     len(char_results),
    'line_count':     len(sentence_lines),
    'avg_confidence': round(float(avg_conf), 2),
    'characters': [
        {
            'letter':     c['letter'],
            'confidence': c['confidence'],
            'low_conf':   c['low_conf'],
            'top_k':      c['top_k'],
            'line_idx':   c['line_idx'],
            'x1': c['x1'], 'x2': c['x2'],
            'y1': c['y1'], 'y2': c['y2'],
        }
        for c in char_results
    ]
}

with open(f'{OUTPUT_DIR}/ocr_result.json', 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print('═' * 60)
print('  PIPELINE TEST COMPLETE')
print('═' * 60)
print(f'  Text          : {full_text}')
print(f'  Lines         : {len(sentence_lines)}')
print(f'  Characters    : {len(char_results)}')
print(f'  Avg confidence: {avg_conf:.1f}%')
print('─' * 60)
print('  Output files saved to /kaggle/working/test_results/')
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'    {f:<35} {size/1024:.1f} KB')
print('═' * 60)